# Week 2 • Upgraded Lab 02 — Statistical evidence in a real Hubble diagram

**MSc Astrostatistics for Observational Cosmology** • 4 h practical • **Graded Lab 1 (5%)**

**Public observational source:** [Pantheon+ / SH0ES DataRelease](https://github.com/PantheonPlusSH0ES/DataRelease). All astronomical objects analysed here are genuine release entries. Bootstrap and permutation replicates are **not new observed SNe**.

**Main problem:** Is an apparent difference between low- and high-redshift supernova Hubble residuals robust to sampling uncertainty, the redshift boundary, and unusual observations? What *cannot* such a difference establish?

### Learning outcomes
- Distinguish object, measurement, statistic and sampling distribution.
- Respect the physical resampling unit (`CID`) and calibration status.
- Produce effect sizes, bootstrap intervals, empirical CDFs and a conditional permutation diagnostic.
- Identify exchangeability, unequal noise and published covariance as limitations.
- Distinguish a reference cosmology from cosmological parameter estimation.

**Required outputs:** 6 labelled plots; 1 effect-size table; 1 sensitivity table; 300–500-word conclusion. A fixed seed makes results reproducible.

## 0. Before running code: the five-line statistical statement

Write your own answer in the next Markdown cell: **estimand; observations; generative assumptions; uncertainties/dependence/selection; falsification diagnostic**. An example estimand is

$$\Delta_{\mathrm{med}}=\operatorname{median}(r\mid z\ge z_*)-\operatorname{median}(r\mid z<z_*).$$

This is a **descriptive effect**, not a fitted value of $H_0$ or $\Omega_m$.

**Student response (complete before observing results):**

- Estimand: …
- Observed data: …
- Assumptions: …
- Dependence/selection: …
- Diagnostic: …

In [ ]:
# Run once. The public source must be reachable by Colab; no invented fallback observations.
import sys, io
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
from scipy import stats
from scipy.integrate import cumulative_trapezoid

SEED = 2026
rng = np.random.default_rng(SEED)
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.18})
print('Python', sys.version.split()[0], '| NumPy', np.__version__, '| SciPy', __import__('scipy').__version__)

## 1. Download and audit the released Pantheon+ data

The file includes **light-curve measurements**, not necessarily one independent row per physical SN. Inspect `CID` before resampling. We use `MU_SH0ES` only as a pre-calibrated, descriptive distance-modulus product; its absolute calibration means it cannot be used here as an independent $H_0$ measurement.

For a careful reference comparison, distinguish the Hubble-diagram redshift $z_{\rm HD}$ (integral) from heliocentric $z_{\rm HEL}$ (luminosity-distance prefactor).

In [ ]:
BASE = ('https://raw.githubusercontent.com/PantheonPlusSH0ES/DataRelease/'
        'main/Pantheon%2B_Data/4_DISTANCES_AND_COVAR/')
TABLE_URL = BASE + 'Pantheon%2BSH0ES.dat'
CACHE = Path('data_cache/pantheon_plus')
CACHE.mkdir(parents=True, exist_ok=True)
table_path = CACHE / 'Pantheon+SH0ES.dat'
if not table_path.exists():
    resp = requests.get(TABLE_URL, timeout=120)
    resp.raise_for_status()
    table_path.write_bytes(resp.content)
sn = pd.read_csv(table_path, sep=r'\s+')
required = {'CID', 'zHD', 'zHEL', 'MU_SH0ES', 'MU_SH0ES_ERR_DIAG', 'IS_CALIBRATOR'}
missing = required - set(sn.columns)
assert not missing, f'Missing Pantheon+ columns: {missing}'
print('Release measurement rows:', len(sn), '| Unique physical CID:', sn.CID.nunique())
print('Duplicate CID measurement rows:', len(sn)-sn.CID.nunique())
display(sn[list(required)].head())

### Discussion A — what is the statistical unit?

A bootstrap that resamples rows can repeatedly include the same physical SN with different observation identifiers. If the intended sampling unit is a physical SN, that can exaggerate the effective sample size. **For exploratory descriptions only**, aggregate repeated rows by `CID` using medians. This is *not* the released cosmological likelihood and does not preserve the full covariance.

In [ ]:
obs = (sn[['CID', 'zHD', 'zHEL', 'MU_SH0ES', 'MU_SH0ES_ERR_DIAG', 'IS_CALIBRATOR']]
       .replace([np.inf, -np.inf], np.nan)
       .dropna(subset=['CID','zHD','zHEL','MU_SH0ES'])
       .groupby('CID', as_index=False)
       .agg(zHD=('zHD','median'), zHEL=('zHEL','median'),
            MU_SH0ES=('MU_SH0ES','median'),
            MU_ERR_DIAG=('MU_SH0ES_ERR_DIAG','median'),
            IS_CALIBRATOR=('IS_CALIBRATOR','max')))
obs = obs.loc[(obs.zHD > 0.01) & (obs.IS_CALIBRATOR == 0) &
              (obs.zHEL > 0)].copy().reset_index(drop=True)
assert len(obs)>100, 'Unexpectedly small real-data selection; inspect release/filters.'
print('Exploratory Hubble-flow objects:', len(obs))
print('z range:', obs.zHD.min(), obs.zHD.max())

## 2. Define a reference, not a fit

Under spatially flat $\Lambda$CDM (neglecting radiation in this redshift range),

$$E(z)=\sqrt{\Omega_m(1+z)^3+1-\Omega_m},\qquad
D_L=\frac{c}{H_0}(1+z_{\rm HEL})\int_0^{z_{\rm HD}}\!\frac{dz'}{E(z')}.$$

$$\mu_{\rm ref}=5\log_{10}[D_L/\mathrm{Mpc}]+25,\qquad
r_i=\mu_{\rm SH0ES,i}-\mu_{\rm ref,i}.$$

Set $H_0^{\rm ref}=73\,\mathrm{km\ s^{-1}\ Mpc^{-1}}$ and $\Omega_m^{\rm ref}=0.3$. These are **chosen plotting values**, not inferences.

In [ ]:
C_KMS = 299792.458

def mu_flat(z_hd, z_hel, omega_m=0.30, H0=73.0, n_grid=8192):
    """Flat-LCDM SN distance modulus: zHD in integral, zHEL in prefactor."""
    z_hd = np.asarray(z_hd, float)
    z_hel = np.asarray(z_hel, float)
    assert z_hd.shape == z_hel.shape
    assert np.all(z_hd>0) and np.all(z_hel>0)
    assert 0.0 <= omega_m <= 1.0 and H0>0
    grid = np.linspace(0, max(float(np.max(z_hd))*1.002, .02), n_grid)
    ez = np.sqrt(omega_m*(1+grid)**3 + (1-omega_m))
    integral = cumulative_trapezoid(1/ez, grid, initial=0.)
    dc = C_KMS/H0 * np.interp(z_hd, grid, integral)
    dl = (1+z_hel)*dc
    return 5*np.log10(dl)+25

H0_REF, OM_REF = 73., .30
obs['MU_REF'] = mu_flat(obs.zHD, obs.zHEL, OM_REF, H0_REF)
obs['RESID'] = obs.MU_SH0ES-obs.MU_REF
print('Median reference residual [mag]:', round(obs.RESID.median(), 4))
assert np.isfinite(obs.RESID).all()

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 8), sharex=True,
                                     gridspec_kw={'height_ratios':[2,1]}, constrained_layout=True)
ix = np.argsort(obs.zHD.to_numpy())
ax1.scatter(obs.zHD, obs.MU_SH0ES, s=8, alpha=.45, label='Pantheon+ object summaries')
ax1.plot(obs.zHD.to_numpy()[ix], obs.MU_REF.to_numpy()[ix], color='black', lw=1.5, label='Reference flat LCDM')
ax1.set_ylabel('Distance modulus [mag]'); ax1.legend()
ax2.scatter(obs.zHD, obs.RESID, s=9, alpha=.5)
ax2.axhline(0, color='black', lw=1)
ax2.set(xlabel='Hubble-diagram redshift zHD', ylabel='Residual [mag]', xscale='log')
ax1.set_xscale('log')
plt.show()

**Question 1:** Identify two observational effects besides cosmological evolution that could create apparent redshift-dependent residual structure. Explain why the reference model and SH0ES zero-point choice matter.

## 3. Sampling distribution and bootstrap of the median

For $N$ independent, identically sampled *objects*, a nonparametric bootstrap draws $N$ object indices **with replacement** and recomputes an estimator. The resulting distribution approximates resampling uncertainty **conditional on the observed sample and its sampling assumptions**. It does not reproduce survey-selection uncertainty, common calibration errors, or inter-SN covariance. With small or strongly structured samples, ordinary percentile intervals may have inadequate coverage.

In [ ]:
def bootstrap_stat(x, statistic=np.median, n_boot=2500, seed=2026):
    x = np.asarray(x, float)
    assert np.all(np.isfinite(x)) and x.size>1
    local = np.random.default_rng(seed)
    out = np.empty(n_boot)
    for k in range(n_boot):
        out[k] = statistic(local.choice(x, size=x.size, replace=True))
    return out

values = obs.RESID.to_numpy()
boot_med = bootstrap_stat(values, n_boot=3000)
ci_global = np.quantile(boot_med, [.025, .5, .975])
print('Observed median:', np.median(values), ' | Bootstrap 95% percentile CI:', ci_global[[0,2]])
fig, ax = plt.subplots(figsize=(7,3.7))
ax.hist(boot_med, bins=45, color='steelblue', alpha=.8)
ax.axvline(np.median(values), color='black', label='Observed median')
for q in [ci_global[0], ci_global[2]]: ax.axvline(q, ls='--', color='firebrick')
ax.set(xlabel='Bootstrap median residual [mag]', ylabel='Replicates', title='Object-level bootstrap')
ax.legend(); plt.show()

**Question 2:** Which uncertainty components are reflected in this interval, and which are missing? Would a narrow interval prove cosmological-model accuracy?

## 4. Define a prespecified low/high-redshift contrast

Set $z_*=0.30$ **before** inspecting its statistical significance. The estimand is

$$\Delta_{\mathrm{med}}=\mathrm{median}(r_{\mathrm{high}})-\mathrm{median}(r_{\mathrm{low}}).$$

The redshift groups need not have identical scatter or selection functions. A signed effect and interval are more informative than a p-value alone.

In [ ]:
Z_SPLIT = .30

def groups_at(split, frame=obs):
    lo = frame.loc[frame.zHD < split, 'RESID'].to_numpy(float)
    hi = frame.loc[frame.zHD >= split, 'RESID'].to_numpy(float)
    if min(len(lo),len(hi)) < 20: raise ValueError('Insufficient objects in one group.')
    return lo, hi

def delta_median(lo,hi): return float(np.median(hi)-np.median(lo))

def boot_group_delta(lo,hi,n_boot=3000,seed=2026):
    local = np.random.default_rng(seed)
    result=np.empty(n_boot)
    for k in range(n_boot):
        a=local.choice(lo,len(lo),replace=True)
        b=local.choice(hi,len(hi),replace=True)
        result[k]=delta_median(a,b)
    return result

low, high = groups_at(Z_SPLIT)
delta_obs = delta_median(low, high)
boot_delta = boot_group_delta(low,high)
ci_delta=np.quantile(boot_delta,[.025,.975])
print(f'N low={len(low)}, N high={len(high)}')
print(f'Delta median={delta_obs:+.4f} mag, conditional bootstrap 95% CI = ({ci_delta[0]:+.4f}, {ci_delta[1]:+.4f})')
fig,ax=plt.subplots(figsize=(7,3.8))
ax.hist(boot_delta, bins=40, alpha=.8)
ax.axvline(delta_obs,color='black', label='Observed')
ax.axvline(0,color='firebrick',ls='--', label='Zero difference')
ax.set(xlabel='High-z minus low-z median [mag]',ylabel='Bootstrap replicates')
ax.legend();plt.show()

## 5. Permutation testing: useful **only if exchangeability is plausible**

We test a *stronger null* than equal medians: under the test null, low/high labels are exchangeable for the residual measurements. With unequal redshift-dependent errors, selection, or survey composition, this is generally questionable; the resulting p-value is therefore a **conditional diagnostic**, *not* a formal claim about evolution or cosmology.

For $B$ shuffled samples,
$$p_{\mathrm{MC}}=\frac{1+\#\{T_b\ge T_{\rm obs}\}}{B+1}, \quad T=|\Delta_{\rm med}|.$$

**Think before running:** How would unequal group variances invalidate the exchangeability argument?

In [ ]:
def conditional_permutation(lo,hi,n_perm=5000,seed=2026):
    local=np.random.default_rng(seed)
    pool=np.r_[lo,hi]
    n=len(lo)
    observed=abs(delta_median(lo,hi))
    null=np.empty(n_perm)
    for b in range(n_perm):
        perm=local.permutation(pool)
        null[b]=abs(delta_median(perm[:n],perm[n:]))
    p=(1+np.count_nonzero(null>=observed))/(1+n_perm)
    return observed,p,null

T_obs, p_cond, null=conditional_permutation(low, high)
print(f'Conditional-exchangeability p-value: {p_cond:.4g} (not a cosmological model test)')
fig,ax=plt.subplots(figsize=(7,3.8))
ax.hist(null,bins=40,alpha=.8)
ax.axvline(T_obs,color='firebrick',lw=2,label='Observed |effect|')
ax.set(xlabel='Absolute difference in medians [mag]',ylabel='Permuted label assignments')
ax.legend();plt.show()

### Sampling distributions differ by redshift

Plot group histograms and their ECDFs. A visible difference in width or tails is evidence **against** naive exchangeability and a reason to avoid treating the conditional permutation p-value as a definitive inferential result.

In [ ]:
def ecdf(x):
    xs=np.sort(np.asarray(x,float)); return xs, np.arange(1,len(xs)+1)/len(xs)
fig,(ax0,ax1)=plt.subplots(1,2,figsize=(11,4),constrained_layout=True)
for arr,lab in [(low,'Low z'),(high,'High z')]:
    ax0.hist(arr,bins=35,alpha=.50,density=True,label=f'{lab}, N={len(arr)}')
    x,y=ecdf(arr); ax1.step(x,y,where='post',label=lab)
ax0.set(xlabel='Residual [mag]',ylabel='Density',title='Group distributions')
ax1.set(xlabel='Residual [mag]',ylabel='ECDF',title='Empirical CDFs')
ax0.legend();ax1.legend();plt.show()
print('Group std devs:', np.std(low,ddof=1), np.std(high,ddof=1))

**Question 3:** Explain why the statement “p is small, therefore the Universe evolves differently” would be invalid. What would a full correlated-covariance and selection-aware model require?

## 6. Robust summaries and a sensitivity exercise

Use $\mathrm{MAD}=\mathrm{median}(|r_i-\mathrm{median}(r)|)$, and distinguish it from $1.4826\times\mathrm{MAD}$ (a Gaussian-consistent scale estimate).

Reassess the **effect size and its interval only** at nearby prespecified boundaries $0.25$ and $0.35$. Do not scan many boundaries to select the smallest p-value.

In [ ]:
def summary(x):
    x=np.asarray(x,float)
    return dict(N=len(x),mean=np.mean(x),std=np.std(x,ddof=1),
                median=np.median(x),MAD=np.median(np.abs(x-np.median(x))))
summary_table=pd.DataFrame({'All':summary(values),'Low z':summary(low),'High z':summary(high)}).T
print('Robust versus classical summary');display(summary_table.round(5))
rows=[]
for split in [.25,.30,.35]:
    a,b=groups_at(split)
    interval=np.quantile(boot_group_delta(a,b, n_boot=2500, seed=2026),[.025,.975])
    rows.append(dict(z_split=split,N_low=len(a),N_high=len(b),
                     delta_median=delta_median(a,b),ci_low=interval[0],ci_high=interval[1]))
sensitivity=pd.DataFrame(rows)
print('Prespecified boundary sensitivity');display(sensitivity.round(5))

## 7. Influence diagnostic

Identify the five largest $|r_i|$ and compute the effect with/without them. **This is a sensitivity diagnostic, not a license to remove inconvenient observations.**

In [ ]:
extremes=obs.loc[obs.RESID.abs().nlargest(5).index,['CID','zHD','RESID']]
display(extremes)
reduced=obs.drop(index=extremes.index)
lo2,hi2=groups_at(Z_SPLIT, reduced)
influence=pd.DataFrame([
    {'sample':'All objects','N':len(obs),'median_resid':np.median(values),'delta_med':delta_obs},
    {'sample':'Exclude 5 |r| extremes','N':len(reduced),'median_resid':reduced.RESID.median(),'delta_med':delta_median(lo2,hi2)}
])
display(influence)

## 8. Failure tests (run and interpret)

**Counterfactual A:** Change reference $H_0$ from 73 to 70. The *absolute* residual zero point shifts, while the low-minus-high contrast should barely respond to a uniform zero-point shift. Verify numerically.

**Counterfactual B:** Shuffle the *object IDs*, rather than the physical redshifts, and ask whether that constructs a valid astronomical experiment (it does not).

**Counterfactual C:** Explain the difference between drawing bootstrap replicates and adding new cosmological observations.

In [ ]:
shifted=obs.MU_SH0ES.to_numpy()-mu_flat(obs.zHD,obs.zHEL,.30,70.)
diff_shifted = np.median(shifted[obs.zHD>=Z_SPLIT])-np.median(shifted[obs.zHD<Z_SPLIT])
expected_constant = 5*np.log10(70./73.)
print('Median residual shift (reference H0 73 -> 70):',np.median(shifted-values))
print('Theoretical uniform shift:', expected_constant)
print('Difference in redshift contrast:',diff_shifted-delta_obs)
assert np.allclose(shifted-values, expected_constant, atol=3e-4)
assert np.allclose(diff_shifted, delta_obs, atol=3e-4)

# Extended analysis • submit the same notebook (graded 5%)

**E1. Scientific inference (20%)** State your observational and sampling model explicitly.

**E2. Uncertainty (25%)** Report the high-minus-low median with a 95% conditional bootstrap interval, and explain which noise/dependence components are excluded.

**E3. Test validity (20%)** Evaluate whether label exchangeability is credible and name a safer alternative to infer a redshift trend (e.g., a heteroscedastic regression model with the published covariance and survey/selection terms).

**E4. Robustness (20%)** Discuss the three redshift boundaries, robust/classical summary table, and five-object influence analysis.

**E5. Scientific conclusion (15%)** Write 300–500 words supported by the computed figures and numbers.

> Final sentence: “From this lab alone, we can conclude ________, but we cannot conclude ________.”

### Deliverables and reproducibility
- Run from a clean kernel. Include code-generated tables and six required figure panels.
- Preserve real source attribution and random seed. Do not imply bootstrap datasets are observed surveys.
- No inference of $H_0$, claims of evolving dark energy, or rejection of $\Lambda$CDM from the conditional test in this lab.

### Recommended reading
- Pantheon+ / SH0ES public release: https://github.com/PantheonPlusSH0ES/DataRelease
- Course lecture: `lectures/W02_DETAILED_INSTRUCTOR_LECTURE.ipynb`
- Crete hypothesis testing: https://github.com/lionandjelka/2025_summer_school/blob/main/02_Hypothesis_Testing/Hypothesis.ipynb
- For a release-specific cosmological likelihood, consult the published Pantheon+ likelihood implementation before making parameter claims.